# Multimodal Document Analysis with Claude

Analyzes scanned documents, diagrams, and title blocks using Claude via Databricks serving endpoints. Extracts structured data (tags, equipment names, landowners, royalties) and chains with cheaper models for secondary extraction.

In [0]:
%pip install OpenAI
%restart_python

Most companies just want an easy way to analyze scanned documents. Multimodal models unlock this - sure they are expensive, but they are really efficient to 'program'.

In [ ]:
def analyze_image(image_path: str, prompt: str, response_format: dict | None = None) -> str:
    """Encode an image and send it to Claude for analysis."""
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")

    media_type = "image/jpeg"
    if image_path.endswith(".webp"):
        media_type = "image/webp"
    elif image_path.endswith(".png"):
        media_type = "image/png"

    kwargs = dict(
        messages=[
            {"role": "system", "content": "You are an AI assistant that can extract and analyze text from images."},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{image_data}"}},
            ]},
        ],
        model="databricks-claude-3-7-sonnet",
        max_tokens=512,
    )
    if response_format:
        kwargs["response_format"] = response_format

    return client.chat.completions.create(**kwargs).choices[0].message.content


# Analyze three different document types
results = {}

results["pid_tags"] = analyze_image(
    "/Volumes/shm/pid/alb_processed_pdfs/08392ebc7912acbf13066326e7974b28/tiles/08392ebc7912acbf13066326e7974b28_p1_t4.jpg",
    "Extract all the tags and equipment names from this image",
    response_format={"type": "json_object"},
)
print("=== P&ID Tags ===")
print(results["pid_tags"])

results["diagram"] = analyze_image(
    "/Volumes/shm/multimodal/images/handwritten_diagram.jpeg",
    "Describe this diagram with a list of all the boxes",
)
print("\n=== Diagram ===")
print(results["diagram"])

results["title_block"] = analyze_image(
    "/Volumes/shm/multimodal/images/title_block.webp",
    "Extract all the information from this title block",
)
print("\n=== Title Block ===")
print(results["title_block"])

# Keep last result as parsed_text for downstream cells
parsed_text = results["title_block"]

In [ ]:
CATALOG = "shm"  # TODO: update to your catalog
SCHEMA = "default"

table = spark.createDataFrame(
  [(image_path, parsed_text)],
  ['path','parsed_text']
)
table.write.mode('overwrite').format('delta').saveAsTable(f'{CATALOG}.{SCHEMA}.parsed_text')

Now let's use a cheaper model to extract key information

In [ ]:
%sql
-- TODO: update catalog.schema to match CATALOG/SCHEMA above
SELECT 
  *, 
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify all landowners in this document. Return only the names in a comma separated list, without any preamble:', parsed_text)
  ) as landowners, 
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT('Identify gas and oil royalties in the document', parsed_text),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "royalties",
        "schema": {
          "type": "object",
          "properties": {
            "gas_royalty": {"type": "number"},
            "oil_royalty": {"type": "number"}
          }
        }
      }
    }'
  ) as royalties
FROM shm.default.parsed_text